In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import*
spark=SparkSession.builder.appName("case_study_2_aeroflux").getOrCreate()

In [0]:
### Step 1: As this is near real-time streaming scenario, so to build our incremental, and low latency data pipeline, we shall use AutoLoader to read our json files.After reading our stream data and adding some relevant columns, we will then write our data to the external location

bronze_df = (spark.readStream.format("cloudFiles").option("cloudFiles.format", "json").option("cloudFiles.inferColumnTypes", "true").option("cloudFiles.schemaEvolutionMode", "addNewColumns").option("cloudFiles.schemaLocation","/Volumes/aerosurf_case_study_4/bronze_layer/schema_locations/raw_data_schema/").load("/Volumes/aerosurf_case_study_4/bronze_layer/raw_data"))

bronze_df.writeStream.format("delta").option("checkpointLocation","/Volumes/aerosurf_case_study_4/bronze_layer/checkpoint_locations/bronze_checkpoint/").option("cloudFiles.schemaLocation","/Volumes/aerosurf_case_study_4/bronze_layer/schema_locations/final_data_schema/").option("mergeSchema", "true").trigger(availableNow=True).toTable("aerosurf_case_study_4.bronze_layer.incoming_bronze")

### Observation: In the below schema we can see that the schema drift has been handlded by the Autoloader by adding the extra columns named as : water_temp and lidar_depth. Our schema evolution strategy and schema location parameters helped us to handle the discrepancy. 

### SchemaEvolution mode helps us to handle the metadata updates in the schema while ingesting the data and the mergeschema option is used in the WRITE layer so that new columns can be handled while writing our data in delta format

In [0]:
#### Step 2: Let us check how our streaming table looks like and read it in a proper format with the column names. Once we have fixed the data types, arrangement and nomenclature of the columns, this is the final version fo the table we would want to work with.

bronze_df_check= spark.readStream.format("delta").table("aerosurf_case_study_4.bronze_layer.incoming_bronze")

bronze_df_check= (bronze_df_check.selectExpr("drone_id", "cast(battery as int) as battery", "risk_level", "sector", "cast(ts as timestamp) as timestamp", "cast(water_temp as decimal(4,2)) as water_temp"))

display(bronze_df_check, checkpointLocation="/Volumes/aerosurf_case_study_4/bronze_layer/checkpoint_locations/bronze_checkpoint")

In [0]:
#### Step 3: Next we have to create the silver table and add the relevant scd type columns 
spark.sql("""create table if not exists aerosurf_case_study_4.silver_layer.incoming_silver(drone_id string, battery int, risk_level string, sector string,timestamp timestamp,water_temp decimal(4,2), start_date timestamp, end_date timestamp, is_active string) using delta""")

### After building our table we have to register it as a delta table object so we can start performing our merge operations
from delta.tables import DeltaTable
silver_table= DeltaTable.forName(spark,"aerosurf_case_study_4.silver_layer.incoming_silver")

### Here we are using the pysaprk dataframe creation so that we can apply our MERGE conditions
silver_df= spark.read.format("delta").table("aerosurf_case_study_4.silver_layer.incoming_silver")
silver_df.show()

In [0]:
#### Step 4: Now we have to identify which are the records that got changed and needs to be inserted as new updated versions of the old records

## We need to filter out only those records that are currently active because we dont want to make a join unecessarily with old records. Moreover we will select only those columns from the silver table that are required for the join procedure. This early filtering of the data will help us to optimize our JOIN condition

silver_df_active=silver_df.filter(f.col("is_active")==f.lit("Yes")).select(f.col("drone_id"),f.col("battery"),f.col("risk_level"),f.col("sector"),f.col("timestamp"),f.col("water_temp")).selectExpr("src.drone_id","src.battery","src.risk_level","src.sector","src.timestamp","src.water_temp")
### NOTE: We need to filter out the ambiguous column names because of the inner join, only selecting the columns we got because of the source

changed_records= bronze_df_check.alias("src").join(silver_df_active.alias("tgt"),on=(f.col("src.drone_id")==f.col("tgt.drone_id")) & (f.col("src.risk_level")!=f.col("tgt.risk_level")), how="inner")

changed_records=changed_records.withColumn("start_date",f.current_timestamp()).withColumn("end_date",f.lit("9999-12-31").cast("timestamp")).withColumn("is_active",f.lit("Yes"))


In [0]:
### Step 4: Next step is to maintain a SCD Type 2 logic for the "Sector Status" as issued by the legal team. 

#### REQUIREMENT 1: If the risk_level changes then the history must be preserved as we need to know exactly when the beaches became dangerous

### SCD Type 2: If the risk_level changes and obviously the timestamp differs, rest all the filed remian same, then we, we have to preserve the history.

def silver_risk_level_scd_type_2(batch_df, batch_id):
    silver_1 = (silver_df.alias("tgt").merge(batch_df.alias("src"),condition= "src.drone_id = tgt.drone_id AND tgt.is_active='Yes'").whenMatchedUpdate(condition="""src.risk_level <> tgt.risk_level AND src.timestamp > tgt.timestamp AND src.battery=tgt.battery AND src.water_temp=tgt.water_temp""",
    set={"is_active": f.lit("No"),"end_date": f.current_timestamp()}).whenNotMatchedInsert(values={"drone_id":"src.drone_id","battery":"src.battery","risk_level":"src.risk_level","sector":"src.sector","timestamp":"src.timestamp","water_temp":"src.water_temp","start_date":f.current_timestamp(),"end_date":f.lit("9999-12-31").cast("timestamp"),"is_active":f.lit("Yes")})).execute()
    return silver_1    

### SCD TYPE 1: When either the battery_level or temperature changes,then we dont have to preserve history, we just have to write the updated version into the existing records

def silver_battery_level_scd_type_1(batch_df,batch_id):
    silver_1_final= (silver_df.alias("tgt").merge(batch_df.alias("src"),condition="src.drone_id=tgt.drone_id AND tgt.is_active='Yes'").whenMatchedUpdate(condition="""src.battery<>tgt.battery OR src.water_temp<>tgt.water_temp""",set={"battery":"src.battery","risk_level":"src.risk_level","sector":"src.sector","timestamp":"src.timestamp","water_temp":"src.water_temp","start_date":"src.end_date","end_date":f.lit("9999-12-31").cast("timestamp")})).execute()
    return silver_1_final

In [0]:
silver_df= spark.read.table("aerosurf_case_study_4.silver_layer.incoming_silver")
silver_df.show()

In [0]:
bronze_df= spark.read.table("aerosurf_case_study_4.bronze_layer.incoming_bronze")
bronze_df.show(3)

In [0]:
dbutils.fs.ls("/Volumes/aerosurf_case_study_4/bronze_layer/raw_data")

In [0]:
spark.read.text("/Volumes/aerosurf_case_study_4/bronze_layer/raw_data").show(truncate=False)

In [0]:
### Step 2: Now we shall read the table from the bronze_layer schema 
bronze_df=spark.read.table("aerosurf_case_study_4.bronze_layer.incoming_bronze").select(f.col("drone_id"),f.col("battery"),f.col("risk_level"),f.col("sector"),f.col("ts").alias("record_timestamp"),f.col("water_temp"),f.col("ingestion_timestamp")).orderBy(f.col("drone_id").asc(),f.col("timestamp").desc())
bronze_df.show()

In [0]:
#### Step 3: The next step is to create our silver_table and push some initial data. Later using the MERGE LOGIC we shall be creating the SCD Type 2 format
spark.sql("""create table silver_df (drone_id string, battery int, risk_level string, sector string, record_timestamp timestamp, water_temp string, ingestion_timestamp timestamp) using delta location 'aerosurf_case_study_4.silver_layer.silver_df';""")